# 📗 부록: 개체에 외부 표준 ID를 연결합니다

**문서에 나온 대상의 Wikidata ID를 찾아 내 노드에 기록합니다.**  

- **입력:** 시연용 LangChain 개체와 실습용 ‘조던’ 후보 3개.
- **할 일:** 이름으로 후보 찾기 → 종류·설명 비교 → ID 선택 또는 보류 → Neo4j 저장.
- **결과:** 선택한 외부 ID와 이유, 또는 선택하지 못한 이유를 저장한 노드.

1~3절은 Python에서 후보를 비교합니다. 4절에서 그 판단 결과를 실습용 Neo4j에 저장합니다.  
LangChain 검색에는 인터넷이 필요하며 API 키와 LLM은 사용하지 않습니다.  


## 1. 내부 ID와 외부 ID를 구분합니다

**개체**는 사람·국가·소프트웨어처럼 구분해서 기록할 대상입니다.  
**Wikidata**는 대상마다 이름·종류·설명을 정리한 공개 **지식베이스(KB)** 입니다.  
문서의 대상과 같은 항목을 고르는 일이 **개체 연결(Entity Linking)** 입니다. Wikidata의 항목 ID는 `Q`로 시작하며 **Q-ID**라고 부릅니다.  

<img src="images/kb_identity.png" width="1000" alt="연결 전후의 같은 노드를 비교합니다. standard_id는 그대로 두고 external_id와 source_kb 속성을 추가합니다.">

| 노드 속성 | 구분하는 것 | LangChain 예 |
|---|---|---|
| `standard_id` | 내 프로젝트의 노드 | `framework:langchain` |
| `external_id` | Wikidata의 항목 | `Q117340550` |
| `source_kb` | 외부 ID를 가져온 지식베이스 | `wikidata` |

이 부록의 **연결**은 같은 노드에 외부 ID와 출처 속성을 추가하는 것입니다.  
`standard_id`를 바꾸거나 Wikidata 항목의 노드·관계를 새로 만들지는 않습니다.  


## 2. LangChain의 설명과 공식 사이트를 대조합니다

이름 검색으로 찾은 항목이 **후보**입니다. 관련 강의·패키지도 검색될 수 있습니다.  
여기서는 **언어 모델 앱 개발 프레임워크인 LangChain 자체**를 찾습니다.  

#### 개체와 조회 도구 준비

`canonical_name`은 대표 이름, `aliases`는 다른 표기, `context`는 문서에서 확인한 설명입니다.  
제공 파일 `kb_lookup.py`는 Wikidata 조회와 응답 정리를 담당합니다.  


In [ ]:
# 학생용과 정답용 모두 같은 제공 파일과 자료를 읽습니다.
import sys
import json
from pathlib import Path
import pandas as pd

material_dir = Path("data").parent
sys.path.insert(0, str(material_dir.resolve()))
from kb_lookup import search_candidates, candidate_rows

local_entity = {
    "standard_id": "framework:langchain",
    "canonical_name": "LangChain",
    "aliases": ["랭체인"],
    "entity_type": "SoftwareFramework",
    "context": "언어 모델 애플리케이션을 만드는 프레임워크",
    "official_website": "https://langchain.com/",
}
display(pd.DataFrame([local_entity]))


#### 이름과 별칭으로 후보 검색

검색어당 최대 5개를 받고 Q-ID 중복은 제거합니다. `id`는 Q-ID, `label`은 이름, `description`은 설명입니다.  
검색 순서는 바뀔 수 있으며 **첫 결과가 정답이라는 뜻은 아닙니다.**  


In [ ]:
# 이름 검색은 비교할 후보를 모으는 단계입니다.
search_terms = [local_entity["canonical_name"]] + local_entity["aliases"]
candidates_by_id = search_candidates(search_terms)
print("검색어:", search_terms, "/ 후보 수:", len(candidates_by_id))
display(pd.DataFrame(candidates_by_id.values()).reindex(columns=["id", "label", "description"]))


#### 종류·설명·공식 사이트 비교

`P`로 시작하는 ID는 속성입니다. 제공 함수는 종류(`P31`)를 `type`, 공식 사이트(`P856`)를 `websites`로 정리합니다.  
`SoftwareFramework`는 프레임워크, ‘소프트웨어’는 더 넓은 종류입니다. 단어가 다르다는 이유만으로 후보를 제외하지 않습니다.  
설명에서 **강의인지 프레임워크 자체인지** 구분하세요. `websites=[]`는 등록된 사이트 정보가 없다는 뜻입니다.  


In [ ]:
# 상세 정보를 받아 한 행이 후보 하나인 비교 표를 만듭니다.
candidate_table = pd.DataFrame(candidate_rows(list(candidates_by_id)))
display(candidate_table)


#### 확인한 후보 선택

[LangChain 항목](https://www.wikidata.org/wiki/Q117340550)은 언어 모델 앱 개발 프레임워크를 설명하며 공식 사이트도 내 기록과 같습니다. 그래서 아래 ID를 선택합니다.  
`linked`는 선택 완료, `link_reason`은 선택 이유입니다.  
`assert`는 후보 ID와 사이트 주소만 검사합니다. **설명의 의미는 사람이 판단합니다.**  


In [ ]:
# 프레임워크 자체를 설명하는 행에서 고른 ID입니다. 검색 1위라는 이유로 고르지 않습니다.
selected_qid = "Q117340550"
assert selected_qid in candidates_by_id, "검색어와 후보 표에서 LangChain을 다시 확인하세요."
selected = candidate_table.set_index("qid").loc[selected_qid]
assert local_entity["official_website"] in selected["websites"], "공식 사이트가 바뀌었는지 확인하세요."

# 내부 ID를 복사해 두고 외부 ID, 출처, 판단 근거를 추가합니다.
demo_link = {
    "standard_id": local_entity["standard_id"],
    "name": local_entity["canonical_name"], "entity_type": "SoftwareFramework",
    "external_id": selected_qid, "source_kb": "wikidata", "link_status": "linked",
    "link_reason": "내 기록과 후보 설명 모두 언어 모델 앱 개발 프레임워크이며 공식 사이트도 같습니다.",
    "review_reason": None,
}
display(pd.DataFrame([demo_link]))


## 3. 문장이 가리키는 Jordan의 항목을 고릅니다

영어 `Jordan`은 국가 **요르단**과 사람 이름 **조던**에 모두 쓰입니다.  
아래 두 사람은 직업이 다릅니다. 문장에 적힌 **종류·직업**과 후보 설명을 비교하세요.  

<img src="images/kb_candidates.png" width="1000" alt="Jordan이 들어간 이름을 가진 국가 요르단, 컴퓨터 과학자 마이클 I. 조던, 농구 선수 마이클 조던을 비교합니다.">

### 🖐️ 함께 따라하기: 같은 표기를 서로 다른 문장에서 판단합니다

#### 실제 후보 3개 읽기

실제 항목을 골라 저장한 아래 **후보 3개 중에서** 판단합니다.  
`original_label`은 영어 이름, `name`과 `description`은 한국어 요약입니다. 검색 순위가 아닙니다.  


In [ ]:
# [제공코드] 조회 날짜와 출처가 있는 저장본으로 같은 후보를 비교합니다.
snapshot = json.loads((material_dir / "data/kb_jordan_candidates.json").read_text(encoding="utf-8"))
jordan_candidates = snapshot["candidates"]
print("조회 날짜:", snapshot["retrieved_at"])
display(pd.DataFrame(jordan_candidates)[["qid", "original_label", "name", "type", "description"]])


#### 실습 A: 농구 선수 선택

**배경:** 학습용 문서 A의 `Jordan is an American basketball player.`(Jordan은 미국의 농구 선수다)에서 Jordan의 항목을 고릅니다.  
**요구사항:**  
- **jordan_qid**에 후보 하나의 Q-ID 문자열을 넣으세요.
- **jordan_reason**에 후보 설명의 어떤 단서가 문장과 맞는지 한 문장으로 쓰세요.

**확인 기준:** 문장의 직업과 후보 설명이 일치해야 합니다. 선택 이유에는 일치한 직업을 적으세요.  

<details><summary>힌트</summary>

```text
접근방법: 사람 후보들의 직업을 비교합니다.
세부구현:
1. 문장의 직업과 일치하는 Q-ID를 고릅니다.
2. 선택한 ID와 일치한 직업을 각각 기록합니다.
```

</details>


In [ ]:
# (1) 후보 표에서 농구 선수의 Q-ID를 찾아 jordan_qid에 넣으세요.

# (2) 원문과 후보가 일치하는 단서를 jordan_reason에 쓰세요.

# 여기에 코드를 작성하세요.


#### 선택 결과 확인

코드는 정답 ID와 이유의 작성 여부를 검사합니다. **농구 선수라는 직업이 일치한다**고 이유를 썼는지는 직접 확인하세요.  


In [ ]:
# [자가채점] 이유의 의미까지 자동 채점하지는 않습니다.
assert jordan_qid == "Q41421", "국가나 컴퓨터 과학자가 아닌 농구 선수 후보를 선택하세요."
assert isinstance(jordan_reason, str) and jordan_reason.strip(), "선택 근거를 한 문장으로 쓰세요."
print("선택:", jordan_qid, "/ 근거:", jordan_reason)


#### 실습 B: 종류와 직업을 알 수 없을 때 판단

<img src="images/kb_decision.png" width="1000" alt="문서 A는 농구 선수라는 단서로 항목을 선택합니다. 문서 B는 국가인지 사람인지 알 수 없어 외부 ID를 정하지 않습니다.">

**보류**는 후보를 선택하지 않고 추가로 필요한 정보를 기록하는 것입니다.  
`review`는 보류, `review_reason`은 사유, `None`은 선택한 ID가 없다는 뜻입니다.  

**배경:** 학습용 문서 B에는 `I looked up Jordan.`(Jordan을 조사했다)만 있습니다. A와 별도 문서이므로 A의 직업 정보를 가져오지 않습니다.  
**요구사항:**  
- **unclear_qid**에는 후보 Q-ID 또는 `None`, **unclear_status**에는 `'linked'` 또는 `'review'` 중 알맞은 값을 넣으세요.
- **unclear_reason**에 판단 근거와 추가로 필요한 단서를 한 문장으로 쓰세요.

**확인 기준:** B의 정보만으로 후보 하나를 고를 수 있는지 판단하고, **더 확인할 종류·직업 정보**를 이유에 적으세요.  

<details><summary>힌트</summary>

```text
접근방법: 후보가 있어도 선택할 단서가 없으면 보류합니다.
세부구현:
1. 선택 여부에 맞는 ID와 상태를 넣습니다.
2. 더 필요한 정보를 기록합니다.
```

</details>


In [ ]:
# (1) unclear_qid와 unclear_status에 판단한 ID와 상태를 넣으세요.

# (2) 어떤 단서가 더 필요한지 unclear_reason에 쓰세요.

# 여기에 코드를 작성하세요.


#### 보류 결과 확인

문서 B에 종류·직업 정보가 없어서 선택하지 못한 것입니다. 후보가 없어서 보류한 경우와 구분하세요.  


In [ ]:
# [자가채점] 보류 상태와 미확정 ID가 함께 있어야 합니다.
assert unclear_qid is None, "같은 대상을 확인하지 못했으므로 외부 ID는 None입니다."
assert unclear_status == "review", "보류 상태는 review로 기록하세요."
assert isinstance(unclear_reason, str) and unclear_reason.strip(), "추가로 필요한 단서를 쓰세요."
print("상태:", unclear_status, "/ 사유:", unclear_reason)


#### 두 판단을 저장할 행으로 정리

A와 B가 같은 대상이라는 증거가 없으므로 내부 ID를 따로 둡니다.  
`UnresolvedEntity`는 종류도 모르는 기록의 실습 라벨입니다. 사용하지 않는 ID·이유는 `None`으로 둡니다.  


In [ ]:
# [제공코드] 확정한 대상과 아직 구분하지 못한 대상을 별도 내부 ID로 유지합니다.
follow_links = [
    {"standard_id": "person:jordan_basketball", "name": "조던", "entity_type": "Person",
     "external_id": jordan_qid, "source_kb": "wikidata", "link_status": "linked",
     "link_reason": jordan_reason, "review_reason": None},
    {"standard_id": "mention:jordan_unknown", "name": "조던", "entity_type": "UnresolvedEntity",
     "external_id": unclear_qid, "source_kb": None, "link_status": unclear_status,
     "link_reason": None, "review_reason": unclear_reason},
]
display(pd.DataFrame(follow_links)[["standard_id", "external_id", "link_status", "link_reason", "review_reason"]])


## 4. 판단 결과를 Neo4j에 저장합니다

`MERGE`는 같은 라벨과 `standard_id`의 노드를 찾고, 없으면 만듭니다.  
`SET`은 Python 사전에 적힌 ID·상태·이유를 그 노드의 속성에 저장합니다.  

#### Neo4j 연결

본 교안의 `.env`를 사용합니다. [실습 가이드](data/../실습_가이드.md)의 접속 설정을 확인하세요.  


In [ ]:
# 연결 판단 결과를 저장할 실습용 Neo4j에 접속합니다.
import os
from urllib.parse import urlsplit
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase

# 현재 작업 폴더부터 상위로 올라가 가장 가까운 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
neo4j_uri = os.environ["NEO4J_URI"]
# driver는 여러 쿼리에서 재사용할 DB 연결 통로입니다. 계정 정보는 출력하지 않습니다.
driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
# 연결 객체 생성만으로 접속 성공이 보장되지 않으므로 지금 서버 접속을 확인합니다.
driver.verify_connectivity()

def run_cypher(query, **params):
    """값을 매개변수로 전달하고 Cypher 결과를 딕셔너리 목록으로 돌려줍니다."""
    # 쿼리마다 세션을 열고 with 블록이 끝나면 닫습니다. driver는 계속 재사용합니다.
    with driver.session() as session:
        # RETURN에서 붙인 별칭이 딕셔너리 키가 되어 파이썬에서 조회할 수 있습니다.
        return [record.data() for record in session.run(query, **params)]

# 주소에 계정 정보가 포함되어 있어도 호스트와 포트만 확인합니다.
connection_address = urlsplit(neo4j_uri)
print("Neo4j 연결 완료. 호스트:", connection_address.hostname, "/ 포트:", connection_address.port)


#### 저장 함수와 LangChain 시연

`save_link(row)`는 사전 1개를 저장하고 **조회 행을 담은 리스트**를 반환합니다.  
`$row`는 전달한 사전, `$($entity_type)`은 지정한 라벨입니다.  
`None`은 Cypher의 `null`로 전달되어 해당 속성을 제거합니다. 함수는 입력값을 저장하며 후보 선택은 하지 않습니다.  


In [ ]:
# 같은 함수를 확정과 보류에 모두 사용합니다. 내부 ID와 원래 라벨은 유지합니다.
def save_link(row):
    """한 개체의 연결 상태를 저장하고 저장된 ID와 근거를 반환합니다."""
    return run_cypher("""
    MERGE (n:$($entity_type) {standard_id: $row.standard_id})
    SET n.name = $row.name, n.entity_type = $entity_type,
        n.external_id = $row.external_id, n.source_kb = $row.source_kb,
        n.link_status = $row.link_status, n.link_reason = $row.link_reason,
        n.review_reason = $row.review_reason
    RETURN n.standard_id AS standard_id, n.external_id AS external_id,
           n.source_kb AS source_kb, n.link_status AS link_status,
           n.link_reason AS link_reason, n.review_reason AS review_reason
    """, entity_type=row["entity_type"], row=row)

display(pd.DataFrame(save_link(demo_link)))


### 🖐️ 함께 따라하기: 두 판단을 저장합니다

#### 실습 C: 저장 후 재조회

**배경:** A에서 선택한 농구 선수와 B에서 보류한 기록을 DB에 각각 저장합니다.  
**요구사항:**  
- **saved_follow**를 리스트로 만들고, `follow_links`의 각 사전을 `save_link`로 저장한 반환 행을 **입력과 같은 순서로** 모으세요.
- **saved_follow**를 DataFrame으로 출력하세요. 저장 셀을 두 번 실행한 뒤 다음 검사 셀을 실행하세요.

**확인 기준:** `saved_follow`는 A, B 순서의 사전 2개입니다. 다음 제공 셀이 DB를 다시 읽어 각 내부 ID에 노드가 하나인지, 저장한 값이 입력과 같은지 검사합니다.  

<details><summary>힌트</summary>

```text
접근방법: 저장 함수의 반환 리스트를 합칩니다.
세부구현:
1. 빈 리스트를 만들고 follow_links를 순회합니다.
2. 반환 행을 extend로 모아 출력합니다.
```

</details>


In [ ]:
# (1) saved_follow를 빈 리스트로 만드세요.

# (2) follow_links의 각 행을 저장하고 반환 행을 saved_follow에 모으세요.

# (3) saved_follow를 DataFrame으로 출력하세요.

# 여기에 코드를 작성하세요.


#### 실제 저장값과 중복 확인

LangChain, A의 농구 선수, B의 미확정 기록을 다시 조회합니다. **총 3행**이며 각 행의 ID·상태·이유가 저장 전 사전과 같아야 합니다.  


In [ ]:
# [자가채점] 현재 실습의 ID만 조회하므로 다른 노드를 검사하지 않습니다.
expected_rows = [demo_link] + follow_links
standard_ids = [row["standard_id"] for row in expected_rows]
stored = run_cypher("""
MATCH (n) WHERE n.standard_id IN $standard_ids
RETURN n.standard_id AS standard_id, n.external_id AS external_id,
       n.source_kb AS source_kb, n.link_status AS link_status,
       n.link_reason AS link_reason, n.review_reason AS review_reason,
       labels(n) AS labels
ORDER BY standard_id
""", standard_ids=standard_ids)
display(pd.DataFrame(stored))

assert isinstance(saved_follow, list) and len(saved_follow) == 2, "두 행의 저장 결과를 saved_follow에 모으세요."
expected_follow = [{key: row[key] for key in ["standard_id", "external_id", "source_kb", "link_status", "link_reason", "review_reason"]}
                   for row in follow_links]
assert saved_follow == expected_follow, "saved_follow에 각 저장 호출이 반환한 행을 순서대로 모으세요."
assert len(stored) == 3, "내부 ID가 같은 노드가 중복되거나 빠졌는지 확인하세요."
stored_by_id = {row["standard_id"]: row for row in stored}
assert set(stored_by_id) == set(standard_ids), "내부 standard_id를 유지하세요."
for expected in expected_rows:
    actual = stored_by_id[expected["standard_id"]]
    assert expected["entity_type"] in actual["labels"], "원래 타입 라벨을 유지하세요."
    for key in ["external_id", "source_kb", "link_status", "link_reason", "review_reason"]:
        assert actual[key] == expected[key], f"저장된 {key} 값을 확인하세요."
print("내부 ID 보존, 연결·보류 상태와 근거 저장을 확인했습니다.")


#### 연결 종료

모든 조회를 마친 뒤 실행합니다.  


In [ ]:
# [제공코드] 다시 DB 실습을 하려면 연결 셀부터 실행합니다.
driver.close()


**이름으로 후보를 찾습니다. 종류·설명으로 대상을 구분하고, 구분할 정보가 없으면 외부 ID를 정하지 않습니다.**  

<details><summary>공식 자료와 조회 도구</summary>

- [Wikidata 항목과 ID](https://www.wikidata.org/wiki/Help:Items)
- [검색 API](https://www.wikidata.org/w/api.php?action=help&modules=wbsearchentities)
- [항목 데이터 조회](https://www.wikidata.org/wiki/Wikidata:Data_access)
- 제공 파일 `kb_lookup.py`: 공개 검색, 속성 조회, 비교 행 생성. 요청 실패는 후보 없음과 구분해 에러로 알립니다.

</details>
